In [0]:
dbutils.widgets.text("catalog_name"," ")
dbutils.widgets.text("folder_id"," ")
catalog_name = dbutils.widgets.get("catalog_name")
folder_id = dbutils.widgets.get("folder_id")
import os

In [0]:
#%run /Workspace/p2p/project2_p2p2/config_parameter

In [0]:


# Access values
print(catalog_name, folder_id)


In [0]:
# Databricks notebook source
%pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [0]:
# Databricks notebook source


from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import os
import io
from datetime import datetime, timezone

SERVICE_ACCOUNT_FILE = f"/Volumes/{catalog_name}/staging/p2p2_files/metadata/service_account.json"
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
VOLUME_BASE_PATH = f"/Volumes/{catalog_name}/staging/p2p2_files/"  # <-- FIXED

creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES)
drive_service = build('drive', 'v3', credentials=creds)

def get_full_path(file_id, drive_service, path_cache={}):
    if file_id in path_cache:
        return path_cache[file_id]
    file = drive_service.files().get(
        fileId=file_id,
        fields='id, name, parents, mimeType'
    ).execute()
    if 'parents' in file:
        parent_path = get_full_path(file['parents'][0], drive_service, path_cache)
        full_path = os.path.join(parent_path, file['name'])
    else:
        full_path = file['name']
    path_cache[file_id] = full_path
    return full_path

def list_all_files(folder_id, drive_service, parent_path=""):
    query = f"'{folder_id}' in parents and trashed = false"
    files = drive_service.files().list(
        q=query,
        fields="files(id, name, mimeType,modifiedTime)"
    ).execute().get('files', [])
    results = []
    for f in files:
        if f['mimeType'] == 'application/pdf' or f['mimeType'] == 'text/csv':
            results.append({
                "id": f['id'],
                "name": f['name'],
                "modifiedTime": f['modifiedTime'],
                "full_path": os.path.join(parent_path, f['name'])
            })
        elif f['mimeType'] == 'application/vnd.google-apps.folder':
            new_path = os.path.join(parent_path, f['name'])
            results.extend(list_all_files(f['id'], drive_service, new_path))
    return results

def read_files_as_binary(file_id):
    request = drive_service.files().get_media(fileId=file_id)
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    buffer.seek(0)
    return buffer.read()

pdf_files = list_all_files(f"{folder_id}", drive_service)
for pdf in pdf_files:
    destination_path = os.path.join(VOLUME_BASE_PATH, pdf['full_path'])
    try:
        file_info = dbutils.fs.ls(destination_path)
        modificationTime = file_info[0].modificationTime
        modificationTime_sink = datetime.fromtimestamp(modificationTime / 1000, tz=timezone.utc)
        modificationTime_source = datetime.strptime(
            pdf['modifiedTime'], "%Y-%m-%dT%H:%M:%S.%fZ"
        ).replace(tzinfo=timezone.utc)
        if modificationTime_source > modificationTime_sink:
            print(f"Downloading: {pdf['full_path']}")
            binary_data = read_files_as_binary(pdf['id'])
            destination_dir = os.path.dirname(destination_path)
            os.makedirs(destination_dir, exist_ok=True)
            with open(destination_path, "wb") as f:
                f.write(binary_data)
                print(f"✅ Saved to: {destination_path}")
        else:
            print(f"No New file arrived in source")
    except Exception as e:
        print(f"File not found: {destination_path}. Downloading new file.")
        binary_data = read_files_as_binary(pdf['id'])
        destination_dir = os.path.dirname(destination_path)
        os.makedirs(destination_dir, exist_ok=True)
        with open(destination_path, "wb") as f:
            f.write(binary_data)
            print(f"✅ Saved to: {destination_path}")

In [0]:
%sql
DESCRIBE HISTORY workspace.default.product_delta